# **_`Pyspark Filter`_**

To limit the data result or extract a subset of master data, use the `filter` method with various conditions:

1. `filter(df.column != 50)` — Filter rows where `column` is not equal to 50.
2. `filter((f.col('column1') > 50) & (f.col('column2') > 50))` — Filter rows where both `column1` and `column2` are greater than 50.
3. `filter((f.col('column1') > 50) | (f.col('column2') > 50))` — Filter rows where either `column1` or `column2` is greater than 50.
4. `filter(df.column.isNull())` — Filter rows where `column` is null.
5. `filter(df.column.isNotNull())` — Filter rows where `column` is not null.
6. `filter(df.column.like('%%'))` — Filter rows where `column` matches a pattern.
7. `filter(df.name.isin())` — Filter rows where `name` is in a specified list.
8. `filter(df.column.contains(''))` — Filter rows where `column` contains a substring.
9. `filter(df.column.startswith(''))` — Filter rows where `column` starts with a substring.
10. `filter(df.column.endswith(''))` — Filter rows where `column` ends with a substring.

### Create DataFrame

In [0]:
# =============================================================================
# CREATE SAMPLE EMPLOYEE DATAFRAME
# =============================================================================
# This cell creates a sample employee dataset to demonstrate various PySpark
# filter operations. The dataset contains 10 employee records with different
# attributes that will be used for filtering examples.
# =============================================================================

# Sample employee data: List of tuples containing employee information
# Structure: (id, name, gender, year, dept, salary, country)
# Note: Employee ID 80 (Emma) has None for gender - used to demonstrate null filtering
employee_data = [
  (10, "John", "M", 2010, 10, 100000, "USA"),
  (20, "Mary", "F", 2011, 20, 200000, "Canada"),
  (30, "Mike", "M", 2012, 30, 300000, "Mexico"),
  (40, "Jane", "F", 2013, 40, 400000, "Brazil"),
  (50, "Tom", "M", 2014, 50, 500000, "USA"),
  (60, "Sara", "F", 2015, 60, 600000, "USA"),
  (70, "Bob", "M", 2016, 70, 700000, "USA"),
  (80, "Emma", None, 2017, 80, 800000, "USA"),  # gender is None (null)
  (90, "Alex", "M", 2018, 90, 900000, "USA"),
  (100, "Lily", "F", 2019, 100, 1000000, "USA")
]

# Define schema: List of column names that will be used for the DataFrame
# This maps each position in the tuple to a named column
schema = ["id", "name", "gender", "year", "dept", "salary", "country"]

# Create DataFrame: spark.createDataFrame() converts the Python list into a Spark DataFrame
# Parameters:
#   - employee_data: The list of tuples containing the data
#   - schema: The list of column names
df = spark.createDataFrame(employee_data, schema)

# Display the DataFrame: display() is a Databricks-specific function that shows
# the DataFrame in an interactive table format with sorting and filtering capabilities
display(df)

In [0]:
# =============================================================================
# FILTER EXAMPLE 1: Range Filtering with AND Condition (&)
# =============================================================================
# This demonstrates filtering rows based on a numeric range using the AND operator.
# We want to keep only employees whose department number is between 10 and 50 (inclusive).
# =============================================================================

# Apply filter with two conditions joined by & (AND operator):
#   Condition 1: df.dept >= 10  (department is greater than or equal to 10)
#   Condition 2: df.dept <= 50  (department is less than or equal to 50)
#
# IMPORTANT: When using multiple conditions in PySpark:
#   - Use & for AND logic (not 'and')
#   - Use | for OR logic (not 'or')
#   - Always wrap each condition in parentheses: (condition1) & (condition2)
#
# Expected result: 5 rows (dept: 10, 20, 30, 40, 50)
df_filtered = df.filter((df.dept >= 10) & (df.dept <= 50))

# Display the filtered DataFrame
display(df_filtered)

In [0]:
# =============================================================================
# FILTER EXAMPLE 2: Multiple Conditions with Mixed Data Types
# =============================================================================
# This demonstrates filtering with AND logic on different data types:
#   - Numeric comparison (salary > 500000)
#   - String equality (country == "USA")
# =============================================================================

# Apply filter with two conditions joined by & (AND operator):
#   Condition 1: df.salary > 500000  (salary is greater than 500,000)
#   Condition 2: df.country == "USA" (country is exactly "USA")
#
# This filter finds high-earning employees (salary > 500k) who work in the USA.
# Both conditions must be true for a row to be included in the result.
#
# Expected result: 4 rows
#   - Sara (60, 600000, USA)
#   - Bob (70, 700000, USA)
#   - Emma (80, 800000, USA)
#   - Alex (90, 900000, USA)
#   - Lily (100, 1000000, USA)
#
# Note: Tom (50, 500000, USA) is NOT included because salary must be GREATER than 500000
display(df.filter((df.salary > 500000) & (df.country == "USA")))

In [0]:
# =============================================================================
# FILTER EXAMPLE 3: OR Condition with Null Value Checking
# =============================================================================
# This demonstrates:
#   1. Checking for null values using isNull() method
#   2. Using OR logic (|) to combine conditions
#   3. String equality comparison
# =============================================================================

# Apply filter with two conditions joined by | (OR operator):
#   Condition 1: df.gender.isNull()  (gender column is null/None)
#   Condition 2: df.gender == "M"    (gender is exactly "M" for Male)
#
# OR logic (|) means if EITHER condition is true, the row is included.
# This filter finds employees who are either male OR have missing gender data.
#
# Key methods for null handling:
#   - isNull(): Returns True if the value is null/None
#   - isNotNull(): Returns True if the value is not null
#
# Expected result: 6 rows
#   - John (10, M)
#   - Mike (30, M)
#   - Tom (50, M)
#   - Bob (70, M)
#   - Emma (80, None)  <- Included because gender is null
#   - Alex (90, M)
#
# Note: All female employees (Mary, Jane, Sara, Lily) are excluded
display(df.filter((df.gender.isNull()) | (df.gender == "M")))

In [0]:
# =============================================================================
# FILTER EXAMPLE 4: String Pattern Matching - startswith()
# =============================================================================
# This demonstrates filtering rows based on string prefix matching.
# We want to find all employees whose names start with the letter "A".
# =============================================================================

# Apply filter using startswith() method:
#   df.name.startswith("A") - Returns True if the name column starts with "A"
#
# String pattern matching methods in PySpark:
#   - startswith(prefix): Checks if string starts with the specified prefix
#   - endswith(suffix): Checks if string ends with the specified suffix
#   - contains(substring): Checks if string contains the specified substring
#   - like(pattern): SQL-style pattern matching with wildcards (%, _)
#
# Case sensitivity: startswith() is case-sensitive by default
#   - startswith("A") will match "Alex" but NOT "alex"
#
# Expected result: 1 row
#   - Alex (90, M, 900000, USA)
display(df.filter(df.name.startswith("A")))

In [0]:
# =============================================================================
# FILTER EXAMPLE 5: String Pattern Matching - startswith() with Multiple Matches
# =============================================================================
# Another example of startswith() filtering, this time for names beginning with "J".
# This demonstrates that startswith() can return multiple rows if multiple names
# match the pattern.
# =============================================================================

# Apply filter using startswith() method:
#   df.name.startswith("J") - Returns True if the name column starts with "J"
#
# Expected result: 2 rows
#   - John (10, M, 100000, USA)
#   - Jane (40, F, 400000, Brazil)
#
# Comparison with previous cell:
#   - Cell 7 filtered for "A" prefix -> 1 result (Alex)
#   - Cell 8 filters for "J" prefix -> 2 results (John, Jane)
#
# This shows that the same method can return different numbers of rows
# depending on the data distribution.
display(df.filter(df.name.startswith("J")))

In [0]:
# =============================================================================
# FILTER EXAMPLE 6: String Pattern Matching - endswith()
# =============================================================================
# This demonstrates filtering rows based on string suffix matching.
# We want to find all employees whose names end with the letter "a".
# =============================================================================

# Apply filter using endswith() method:
#   df.name.endswith("a") - Returns True if the name column ends with "a"
#
# endswith() is the complement to startswith():
#   - startswith(prefix): Checks the beginning of the string
#   - endswith(suffix): Checks the end of the string
#
# Case sensitivity: endswith() is case-sensitive by default
#   - endswith("a") will match "Sara" but NOT "SarA"
#   - To make it case-insensitive, convert to lowercase first:
#     df.filter(lower(df.name).endswith("a"))
#
# Expected result: 3 rows
#   - Emma (80, None, 800000, USA)
#   - Sara (60, F, 600000, USA)
#
# Note: Mary ends with "y" not "a", so it's excluded
display(df.filter(df.name.endswith("a")))